# 01. KV Cache Quantization: KIVI and KVQuant

- **Category:** 14_quantisation / 05_kv_attention_optim_and_benchmarks
- **Lineage & Citations:** Liu et al. (2024, *KIVI: A Tuning-Free Asymmetric 2bit Quantization for KV Cache*, ICML); Hooper et al. (2024, *KVQuant: Towards 10 Million Context Length LLM Inference with KV Cache Quantization*).
- **Purpose:** Compress long-context LLM Key-Value caches down to 2-bit by exploiting the orthogonal outlier structure of Keys (per-channel across time) vs. Values (per-token across channels) with a sliding-window full-precision buffer.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. QLoRA LoftQ and QA-LoRA Quantized PEFT](../04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. Quantized Attention SageAttention INT8 FP8](02_quantized_attention_sageattention_int8_fp8.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT quantize Keys per-token across channels (Key outlier channels will artificially expand the quantization range of normal channels); do NOT omit the recent-token sliding window buffer (immediate past tokens have not stabilized cross-sequence variance, causing catastrophic attention degradation).

## Mathematical Foundations & Governing Equations

1. **KV Cache Memory Footprint**: $\text{Memory}_{\text{KV}} = 2 \cdot B \cdot L \cdot H \cdot D \cdot S \cdot \text{bytes}$
2. **KIVI Key Quantization (Per-Channel Across Sequence)**:
   $$s_{K, c} = \frac{\max_s K_{s, c} - \min_s K_{s, c}}{2^b - 1}, \quad z_{K, c} = \text{round}\left( \frac{-\min_s K_{s, c}}{s_{K, c}} \right), \quad K_{s, c}^{\text{deq}} = (K_{s, c}^{\text{int}} - z_{K, c}) \cdot s_{K, c}$$
3. **KIVI Value Quantization (Per-Token Across Channels)**:
   $$s_{V, s} = \frac{\max_c V_{s, c} - \min_c V_{s, c}}{2^b - 1}, \quad z_{V, s} = \text{round}\left( \frac{-\min_c V_{s, c}}{s_{V, s}} \right), \quad V_{s, c}^{\text{deq}} = (V_{s, c}^{\text{int}} - z_{V, s}) \cdot s_{V, s}$$
4. **Sliding-Window Composite Attention Decoding**:
   $$\mathbf{K}_{\text{eff}} = [\mathbf{K}_{\text{past}}^{\text{2-bit}} \;;\; \mathbf{K}_{\text{recent}}^{\text{FP32}}], \quad \mathbf{V}_{\text{eff}} = [\mathbf{V}_{\text{past}}^{\text{2-bit}} \;;\; \mathbf{V}_{\text{recent}}^{\text{FP32}}], \quad \mathbf{y} = \text{softmax}\left( \frac{\mathbf{q} \mathbf{K}_{\text{eff}}^\top}{\sqrt{d_k}} \right) \mathbf{V}_{\text{eff}}$$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def quantize_k_per_channel(k: torch.Tensor, bits: int = 2, eps: float = 1e-7):
    qmax = float((1 << bits) - 1)
    k_min = torch.min(k, dim=0, keepdim=True).values; k_max = torch.max(k, dim=0, keepdim=True).values
    scale = torch.clamp((k_max - k_min) / qmax, min=eps)
    zero = torch.round(torch.clamp(-k_min / scale, 0.0, qmax))
    k_int = torch.clamp(torch.round(k / scale + zero), 0.0, qmax)
    return (k_int - zero) * scale, k_int, scale, zero

def quantize_v_per_token(v: torch.Tensor, bits: int = 2, eps: float = 1e-7):
    qmax = float((1 << bits) - 1)
    v_min = torch.min(v, dim=1, keepdim=True).values; v_max = torch.max(v, dim=1, keepdim=True).values
    scale = torch.clamp((v_max - v_min) / qmax, min=eps)
    zero = torch.round(torch.clamp(-v_min / scale, 0.0, qmax))
    v_int = torch.clamp(torch.round(v / scale + zero), 0.0, qmax)
    return (v_int - zero) * scale, v_int, scale, zero

def kivi_attention_decode(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, window_size: int = 4, bits: int = 2):
    seq_len, d_k = k.shape
    if seq_len > window_size:
        k_past_deq, _, _, _ = quantize_k_per_channel(k[: seq_len - window_size], bits=bits)
        v_past_deq, _, _, _ = quantize_v_per_token(v[: seq_len - window_size], bits=bits)
        k_eff = torch.cat([k_past_deq, k[seq_len - window_size :]], dim=0)
        v_eff = torch.cat([v_past_deq, v[seq_len - window_size :]], dim=0)
    else: k_eff, v_eff = k.clone(), v.clone()
    scores = (q @ k_eff.T) * (1.0 / math.sqrt(float(d_k)))
    return torch.softmax(scores, dim=-1) @ v_eff, k_eff, v_eff

In [ ]:
# 1. Linear anchor parity
assert float(torch.max(torch.abs(weights["test_input"] @ weights["weight"] + weights["bias"] - weights["test_output"]))) < 1e-5

# 2. Key per-channel quantization parity
k_deq, k_int, k_s, k_z = quantize_k_per_channel(weights["k"], bits=2)
assert float(torch.max(torch.abs(k_deq - weights["k_deq"]))) < 1e-5
assert float(torch.max(torch.abs(k_int - weights["k_int"]))) < 1e-5
assert float(torch.max(torch.abs(k_s - weights["k_scale"]))) < 1e-5
assert float(torch.max(torch.abs(k_z - weights["k_zero"]))) < 1e-5

# 3. Value per-token quantization parity
v_deq, v_int, v_s, v_z = quantize_v_per_token(weights["v"], bits=2)
assert float(torch.max(torch.abs(v_deq - weights["v_deq"]))) < 1e-5
assert float(torch.max(torch.abs(v_int - weights["v_int"]))) < 1e-5
assert float(torch.max(torch.abs(v_s - weights["v_scale"]))) < 1e-5
assert float(torch.max(torch.abs(v_z - weights["v_zero"]))) < 1e-5

# 4. KIVI Attention Decoding with window buffer parity
out_kivi, k_eff, v_eff = kivi_attention_decode(weights["q"], weights["k"], weights["v"], window_size=4, bits=2)
assert float(torch.max(torch.abs(out_kivi - weights["out_kivi"]))) < 1e-5
assert float(torch.max(torch.abs(k_eff - weights["k_eff"]))) < 1e-5
assert float(torch.max(torch.abs(v_eff - weights["v_eff"]))) < 1e-5
print("All KIVI Key/Value and composite attention assertions verified against golden checkpoint!")

In [ ]:
# Decoding Attention Fidelity Evaluation: KIVI 2-Bit vs FP32 Baseline
y_fp32 = weights["out_fp32"]
mse = float(torch.mean((y_fp32 - weights["out_kivi"]) ** 2))
sig_pwr = float(torch.mean(y_fp32 ** 2))
sqnr = 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

mem_reduction = 32.0 / 2.0
print(f"KIVI 2-Bit Decoding SQNR vs FP32: {sqnr:.2f} dB")
print(f"KV Cache Memory Compression Factor: {mem_reduction:.1f}x (FP32) / {mem_reduction/2.0:.1f}x (FP16)")
assert sqnr > 8.0, "KIVI 2-bit attention with windowing must preserve >8 dB decoding SQNR!"
assert mem_reduction >= 8.0, "2-bit KV compression must yield >=8x memory reduction!"

In [ ]:
# Latency Microbenchmark: KIVI Decoding Attention (SeqLen=1024, HeadDim=64)
s_bench, d_bench = 1024, 64
q_bench = torch.randn(1, d_bench)
k_bench = torch.randn(s_bench, d_bench)
v_bench = torch.randn(s_bench, d_bench)

t0 = time.perf_counter()
for _ in range(50):
    _ = kivi_attention_decode(q_bench, k_bench, v_bench, window_size=32, bits=2)
t_decode = (time.perf_counter() - t0) / 50 * 1e3

print(f"KIVI Attention Decode (S={s_bench}, D={d_bench}, W=32) Latency: {t_decode:.3f} ms")
assert t_decode < 25.0, "KIVI attention decoding must execute in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Asymmetric Directional Quantization**: Key outlier channels remain invariant across time (quantize per-channel across tokens); Value outliers emerge per token (quantize per-token across channels).
2. **Sliding-Window Stability**: Buffering recent tokens in FP16/FP32 guarantees attention fidelity to immediately preceding prompt tokens while historical tokens are compressed $8\times$.
3. **Dense-Sparse Outlier Channel Isolation**: KVQuant isolates the top $1\%\text{--}2\%$ peak Key channels, enabling aggressive 2-bit quantization on the remaining dense representation.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_kv_cache_quantization_kivi_and_kvquant.md](../../../markdown/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.md)
- **PyTorch 2.x Module:** [01_kv_cache_quantization_kivi_and_kvquant.py](../../../pytorch2/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.py)
- **Functional JAX Module:** [01_kv_cache_quantization_kivi_and_kvquant.py](../../../jax/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.py)
- **Native Rust Candle Track:** [01_kv_cache_quantization_kivi_and_kvquant.rs](../../../candle/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.rs)
- **Hardware Kernel (CPU / WGSL):** [01_kv_cache_quantization_kivi_and_kvquant.rs](../../../kernels/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.rs)
- **Golden SafeTensors:** [01_kv_cache_quantization_kivi_and_kvquant.safetensors](../../../weights/14_quantisation/05_kv_attention_optim_and_benchmarks/01_kv_cache_quantization_kivi_and_kvquant.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. QLoRA LoftQ and QA-LoRA Quantized PEFT](../04_vector_lattice_qat_and_lowbit/06_qlora_loftq_and_qa_lora_quantized_peft.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 02. Quantized Attention SageAttention INT8 FP8](02_quantized_attention_sageattention_int8_fp8.ipynb) &rarr;